Import Libraries

In [79]:
import pandas as pd
import numpy as np

Extract

In [80]:
jobs = pd.read_csv ("C:/Users/lenovo/OneDrive/Desktop/Projects/Job Market Intelligence/data/raw/gsearch_jobs (1).csv")

Create Working Copy

In [81]:
jobs_clean = jobs.copy()

 Data Quality Snapshot

In [82]:
print("Rowss:",jobs_clean.shape[0])
print("Columns: ",jobs_clean.shape[1])
print("Duplicate Rows:",jobs_clean.duplicated().sum())
print("Duplicate Job IDs:",jobs_clean.duplicated(subset="job_id").sum())

Rowss: 61953
Columns:  27
Duplicate Rows: 0
Duplicate Job IDs: 3178


Missing Values Report

In [83]:
missing = (
    jobs_clean.isnull()
    .sum()
    .sort_values(ascending = False)
    .to_frame(name="Missing Values")
)
missing.head(10)

,Missing Values
commute_time,61953
salary_yearly,57884
salary_hourly,56053
salary_max,52441
salary_min,52441
salary,51865
salary_pay,51865
salary_rate,51865
salary_standardized,51865
salary_avg,51865


# Transformation 1: Remove Unnecessary Columns
# Business Reason
The following columns do not provide any useful business information:
- Unnamed: 0 → CSV index generated during export
- index → Duplicate row index
- commute_time → Contains 100% missing values
These columns will be removed to simplify the dataset before analysis.

In [84]:
jobs_clean.columns

Index(['Unnamed: 0', 'index', 'title', 'company_name', 'location', 'via',
       'description', 'extensions', 'job_id', 'thumbnail', 'posted_at',
       'schedule_type', 'work_from_home', 'salary', 'search_term', 'date_time',
       'search_location', 'commute_time', 'salary_pay', 'salary_rate',
       'salary_avg', 'salary_min', 'salary_max', 'salary_hourly',
       'salary_yearly', 'salary_standardized', 'description_tokens'],
      dtype='str')

In [85]:
drop_columns = [
    "Unnamed: 0",
    "index",
    "commute_time"
]

jobs_clean = jobs_clean.drop(columns=drop_columns)

jobs_clean.shape



(61953, 24)

In [86]:
jobs_clean = jobs_clean.drop(
    columns=drop_columns,
    errors="ignore"
)

In [87]:
jobs_clean.shape

(61953, 24)

Transformation 2

In [88]:
jobs_clean['work_from_home'].value_counts().head(15)

work_from_home
True    27980
Name: count, dtype: int64

In [89]:
jobs_clean['location'].value_counts().head(15)

location
 Anywhere                  18067
  United States            10011
Anywhere                    9915
United States               5547
Denver, CO                  1062
  Oklahoma City, OK         1028
  Kansas City, MO            840
Kansas City, MO              573
  Jefferson City, MO         550
Colorado Springs, CO         512
  Bentonville, AR            422
Oklahoma City, OK            409
Jefferson City, MO           371
Aurora, CO                   346
  Topeka, KS                 310
Name: count, dtype: int64

In [90]:
jobs_clean['work_from_home'].value_counts(dropna=False)

work_from_home
NaN     33973
True    27980
Name: count, dtype: int64

In [91]:
jobs_clean.columns

Index(['title', 'company_name', 'location', 'via', 'description', 'extensions',
       'job_id', 'thumbnail', 'posted_at', 'schedule_type', 'work_from_home',
       'salary', 'search_term', 'date_time', 'search_location', 'salary_pay',
       'salary_rate', 'salary_avg', 'salary_min', 'salary_max',
       'salary_hourly', 'salary_yearly', 'salary_standardized',
       'description_tokens'],
      dtype='str')

In [92]:
jobs_clean['work_from_home'].fillna(False)

0         True
1        False
2        False
3         True
4        False
         ...  
61948    False
61949    False
61950    False
61951    False
61952    False
Name: work_from_home, Length: 61953, dtype: object

In [93]:
jobs_clean['title'].sample(20, random_state=42)

40894    Senior Financial Analyst / Business Analyst / ...
16954               Data Analytics&Visualization Associate
60849    Business Analyst/Scrum Master - REMOTE- Contra...
16732                                       Data Scientist
35309               Google Analytics 4 Setup and Dashboard
53952                       Marketing Analytics Consultant
46548                               Need actuarial analyst
35325    Lead Data Analyst, Integrated Business Plannin...
50767    2024 Summer Intern: Master's Sam's Club Supply...
23311    Fall Co-Op - Defense and Special Missions Data...
26030                                         Data Analyst
52179                    Enterprise Analytics Data Analyst
2152                         Sr. Supply Chain Data Analyst
2758               (USA) Senior Data Analyst - Data Triage
13075         Healthcare Data scientist Architect (Gen AI)
39848                         Market Intelligence, Analyst
23025                                 Quality Data Analy

# Transformation 2: Create Experience Level
## Business Reason
The job title contains experience information such as Intern, Junior, Senior, and Lead.
Extracting this information into a separate column allows us to analyze:
- Hiring demand by experience level
- Salary by experience level
- Remote opportunities by experience level

In [94]:
def get_experience_level(title):
    title = str(title).lower()
    
    if("intern" in title or "internship" in title or "co-op" in title):
        return "Intern"
    
    elif("junior" in title or "jr" in title):
        return "Junior"
    
    elif("senior" in title or "sr" in title):
        return "Senior"
    
    elif("lead" in title or "principal" in title):
        return "Lead"
    
    elif("manager" in title or "head" in title or "director" in title):
        return "Manager"
    
    else:
        return"Mid"

In [95]:
jobs_clean["experience_level"] = jobs_clean["title"].apply(get_experience_level)

In [96]:
jobs_clean[["title","experience_level"]].sample(15, random_state = 42)

,title,experience_level
40894,Senior Financial Analyst / Business Analyst / ...,Senior
16954,Data Analytics&Visualization Associate,Mid
60849,Business Analyst/Scrum Master - REMOTE- Contra...,Mid
16732,Data Scientist,Mid
35309,Google Analytics 4 Setup and Dashboard,Mid
53952,Marketing Analytics Consultant,Mid
46548,Need actuarial analyst,Mid
35325,"Lead Data Analyst, Integrated Business Plannin...",Lead
50767,2024 Summer Intern: Master's Sam's Club Supply...,Intern
23311,Fall Co-Op - Defense and Special Missions Data...,Intern


In [97]:
jobs_clean['description_tokens'].head(10)

0                    ['tableau', 'r', 'python', 'sql']
1                                                   []
2                                              ['sql']
3                  ['powerpoint', 'excel', 'power_bi']
4           ['powerpoint', 'excel', 'outlook', 'word']
5    ['power_bi', 'aws', 'excel', 'sql', 'mysql', '...
6                     ['python', 'sas', 'sql', 'spss']
7    ['power_bi', 'tableau', 'jira', 'snowflake', '...
8                                                   []
9                                      ['r', 'python']
Name: description_tokens, dtype: str

In [98]:
jobs_clean['description_tokens'].sample(5,random_state = 42)

40894    ['power_bi', 'tableau', 'excel', 'power_bi', '...
16954           ['tableau', 'dart', 'sql', 'r', 'alteryx']
60849                                                   []
16732                                    ['aws', 'python']
35309                                                   []
Name: description_tokens, dtype: str

another ETL transformation.

Step 1: Test One Row

In [99]:
type(jobs_clean['description_tokens'].iloc[0])

str

Step 2: Convert Strings to Lists

In [100]:
import ast

In [101]:
jobs_clean["description_tokens"] = jobs_clean["description_tokens"].apply(
    lambda x:ast.literal_eval(x) if pd.notnull(x) else []
    )

Step 3: Verify

In [102]:
type(jobs_clean["description_tokens"].iloc[0])

list

In [103]:
jobs_clean["description_tokens"].iloc[0]

['tableau', 'r', 'python', 'sql']

In [106]:
jobs_clean.to_csv(
    "C:/Users/lenovo/OneDrive/Desktop/Projects/Job Market Intelligence/data/processed/jobs_clean.csv",
    index=False
)